# Customer Segmentation & Recommendation System
## Capstone Project - Complete Pipeline

This notebook demonstrates the entire customer segmentation pipeline:
1. Data Loading & Cleaning
2. Feature Engineering
3. Outlier Detection
4. Scaling & PCA
5. K-Means Clustering
6. Evaluation Metrics
7. Recommendation Generation
8. Save Models & Artifacts

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

print('✓ All libraries imported successfully')

## 2. Load & Clean Data

In [ ]:
# Load raw data - works for Local, Kaggle, and Colab
if os.path.exists('/kaggle/input/ecommerce-data/data.csv'):
    df = pd.read_csv('/kaggle/input/ecommerce-data/data.csv', encoding='ISO-8859-1')
    print('Loaded from Kaggle')
elif os.path.exists('/content/data.csv'):
    df = pd.read_csv('/content/data.csv', encoding='ISO-8859-1')
    print('Loaded from Colab')
else:
    df = pd.read_csv(r'C:\major_pro\customer_segmentation\data.csv', encoding='ISO-8859-1')
    print('Loaded from Local')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Data summary statistics
print('Data Statistics:')
print(df.describe().T)

In [ ]:
# Check missing values
missing_data = df.isnull().sum()
missing_percentage = (missing_data[missing_data > 0] / df.shape[0]) * 100
print('Missing Values (%)')
print(missing_percentage.sort_values(ascending=False))

In [ ]:
# Clean data
def clean_data(data):
    df_clean = data.copy()
    
    df_clean = df_clean.dropna(subset=['CustomerID', 'Description'])
    print(f'After removing missing CustomerID/Description: {df_clean.shape[0]} rows')
    
    duplicates_before = df_clean.shape[0]
    df_clean.drop_duplicates(inplace=True)
    print(f'Duplicates removed: {duplicates_before - df_clean.shape[0]}')
    
    df_clean['Transaction_Status'] = np.where(
        df_clean['InvoiceNo'].astype(str).str.startswith('C'),
        'Cancelled',
        'Completed'
    )
    
    unique_codes = df_clean['StockCode'].unique()
    anomalous_codes = [
        code for code in unique_codes
        if sum(c.isdigit() for c in str(code)) in (0, 1)
    ]
    df_clean = df_clean[~df_clean['StockCode'].isin(anomalous_codes)]
    
    service_related = ["Next Day Carriage", "High Resolution Image"]
    df_clean = df_clean[~df_clean['Description'].isin(service_related)]
    
    df_clean['Description'] = df_clean['Description'].str.upper()
    df_clean = df_clean[df_clean['UnitPrice'] > 0]
    
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], format='%m/%d/%Y %H:%M')
    df_clean['InvoiceDay'] = df_clean['InvoiceDate'].dt.date
    
    return df_clean

df_clean = clean_data(df)
print(f'\nCleaned dataset shape: {df_clean.shape}')
print('✓ Data Cleaning Completed')

## 3. Feature Engineering

In [ ]:
def create_customer_features(data):
    df = data.copy()
    
    customer_data = df.groupby('CustomerID')['InvoiceDay'].max().reset_index()
    most_recent_date = pd.to_datetime(df['InvoiceDay']).max()
    customer_data['InvoiceDay'] = pd.to_datetime(customer_data['InvoiceDay'])
    customer_data['Days_Since_Last_Purchase'] = (most_recent_date - customer_data['InvoiceDay']).dt.days
    customer_data.drop(columns=['InvoiceDay'], inplace=True)
    
    total_transactions = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
    total_transactions.rename(columns={'InvoiceNo': 'Total_Transactions'}, inplace=True)
    total_products = df.groupby('CustomerID')['Quantity'].sum().reset_index()
    total_products.rename(columns={'Quantity': 'Total_Products_Purchased'}, inplace=True)
    
    customer_data = pd.merge(customer_data, total_transactions, on='CustomerID')
    customer_data = pd.merge(customer_data, total_products, on='CustomerID')
    
    df['Total_Spend'] = df['UnitPrice'] * df['Quantity']
    total_spend = df.groupby('CustomerID')['Total_Spend'].sum().reset_index()
    avg_transaction = total_spend.merge(total_transactions, on='CustomerID')
    avg_transaction['Average_Transaction_Value'] = avg_transaction['Total_Spend'] / avg_transaction['Total_Transactions']
    
    customer_data = pd.merge(customer_data, total_spend, on='CustomerID')
    customer_data = pd.merge(customer_data, avg_transaction[['CustomerID', 'Average_Transaction_Value']], on='CustomerID')
    
    df['Day_Of_Week'] = df['InvoiceDate'].dt.dayofweek
    df['Hour'] = df['InvoiceDate'].dt.hour
    
    days_between = df.groupby('CustomerID')['InvoiceDay'].apply(
        lambda x: (x.diff().dropna()).apply(lambda y: y.days)
    )
    avg_days_between = days_between.groupby('CustomerID').mean().reset_index()
    avg_days_between.rename(columns={'InvoiceDay': 'Avg_Days_Between_Purchases'}, inplace=True)
    
    fav_day = df.groupby(['CustomerID', 'Day_Of_Week']).size().reset_index(name='Count')
    fav_day = fav_day.loc[fav_day.groupby('CustomerID')['Count'].idxmax()][['CustomerID', 'Day_Of_Week']]
    fav_hour = df.groupby(['CustomerID', 'Hour']).size().reset_index(name='Count')
    fav_hour = fav_hour.loc[fav_hour.groupby('CustomerID')['Count'].idxmax()][['CustomerID', 'Hour']]
    
    customer_data = pd.merge(customer_data, avg_days_between, on='CustomerID')
    customer_data = pd.merge(customer_data, fav_day, on='CustomerID')
    customer_data = pd.merge(customer_data, fav_hour, on='CustomerID')
    
    customer_country = df.groupby(['CustomerID', 'Country']).size().reset_index(name='Count')
    main_country = customer_country.sort_values('Count', ascending=False).drop_duplicates('CustomerID')
    main_country['Is_UK'] = main_country['Country'].apply(lambda x: 1 if x == 'United Kingdom' else 0)
    customer_data = pd.merge(customer_data, main_country[['CustomerID', 'Is_UK']], on='CustomerID', how='left')
    
    cancelled = df[df['Transaction_Status'] == 'Cancelled']
    cancel_freq = cancelled.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
    cancel_freq.rename(columns={'InvoiceNo': 'Cancellation_Frequency'}, inplace=True)
    customer_data = pd.merge(customer_data, cancel_freq, on='CustomerID', how='left')
    customer_data['Cancellation_Frequency'] = customer_data['Cancellation_Frequency'].fillna(0)
    customer_data['Cancellation_Rate'] = customer_data['Cancellation_Frequency'] / total_transactions['Total_Transactions']
    
    df['Year'] = df['InvoiceDate'].dt.year
    df['Month'] = df['InvoiceDate'].dt.month
    monthly_spend = df.groupby(['CustomerID', 'Year', 'Month'])['Total_Spend'].sum().reset_index()
    
    seasonal = monthly_spend.groupby('CustomerID')['Total_Spend'].agg(['mean', 'std']).reset_index()
    seasonal.rename(columns={'mean': 'Monthly_Spending_Mean', 'std': 'Monthly_Spending_Std'}, inplace=True)
    seasonal['Monthly_Spending_Std'] = seasonal['Monthly_Spending_Std'].fillna(0)
    
    def calculate_trend(data):
        if len(data) > 1:
            x = np.arange(len(data))
            slope, _, _, _, _ = linregress(x, data)
            return slope
        return 0
    
    trend = monthly_spend.groupby('CustomerID')['Total_Spend'].apply(calculate_trend).reset_index()
    trend.rename(columns={'Total_Spend': 'Spending_Trend'}, inplace=True)
    
    customer_data = pd.merge(customer_data, seasonal, on='CustomerID')
    customer_data = pd.merge(customer_data, trend, on='CustomerID')
    
    customer_data['CustomerID'] = customer_data['CustomerID'].astype(str)
    numeric_columns = customer_data.select_dtypes(include=['number']).columns
    for col in numeric_columns:
        if customer_data[col].isna().any():
            customer_data[col].fillna(customer_data[col].median(), inplace=True)
    
    return customer_data

customer_features = create_customer_features(df_clean)
print(f'Customer features shape: {customer_features.shape}')
print(f'Features created: {list(customer_features.columns)}')
print('✓ Feature Engineering Completed')
customer_features.head()

## 4. Outlier Detection & Removal

In [ ]:
def detect_and_remove_outliers(data):
    customer_data = data.copy()
    customer_data = customer_data.dropna()
    customer_data.reset_index(drop=True, inplace=True)
    
    model = IsolationForest(contamination=0.05, random_state=0)
    features_df = customer_data.iloc[:, 1:].copy()
    features_df = features_df.fillna(features_df.median(numeric_only=True))
    features = features_df.to_numpy()
    
    customer_data['Outlier_Scores'] = model.fit_predict(features)
    customer_data['Is_Outlier'] = [1 if x == -1 else 0 for x in customer_data['Outlier_Scores']]
    
    outlier_percentage = (customer_data['Is_Outlier'].value_counts(normalize=True) * 100)
    print('Outlier Distribution (%)')
    print(outlier_percentage)
    
    outliers_data = customer_data[customer_data['Is_Outlier'] == 1]
    customer_data_cleaned = customer_data[customer_data['Is_Outlier'] == 0]
    customer_data_cleaned = customer_data_cleaned.drop(columns=['Outlier_Scores', 'Is_Outlier'])
    customer_data_cleaned.reset_index(drop=True, inplace=True)
    
    return customer_data_cleaned, outliers_data

customer_data_cleaned, outliers_data = detect_and_remove_outliers(customer_features)
print(f'\nRemaining rows: {customer_data_cleaned.shape[0]}')
print(f'Outliers removed: {outliers_data.shape[0]}')
print('✓ Outlier Removal Completed')

## 5. Scaling & PCA

In [ ]:
# Global variables to store scaler and pca for pkl saving later
scaler = None
pca = None

def scale_and_apply_pca(data):
    global scaler, pca
    
    customer_data = data.copy()
    
    numeric_columns = customer_data.select_dtypes(include=['number']).columns
    for col in numeric_columns:
        if customer_data[col].isna().any():
            customer_data[col].fillna(customer_data[col].median(), inplace=True)
    
    columns_to_exclude = ['CustomerID', 'Is_UK', 'Day_Of_Week']
    columns_to_scale = customer_data.columns.difference(columns_to_exclude)
    
    customer_data_scaled = customer_data.copy()
    scaler = StandardScaler()
    customer_data_scaled[columns_to_scale] = scaler.fit_transform(customer_data_scaled[columns_to_scale])
    
    customer_data_scaled.set_index('CustomerID', inplace=True)
    
    pca_full = PCA().fit(customer_data_scaled)
    explained_variance_ratio = pca_full.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    print('Cumulative Explained Variance by Components:')
    for i, var in enumerate(cumulative_variance[:10], 1):
        print(f'PC{i}: {var:.4f}')
    
    optimal_k = 6
    pca = PCA(n_components=optimal_k)
    customer_data_pca = pca.fit_transform(customer_data_scaled)
    customer_data_pca = pd.DataFrame(customer_data_pca, columns=[f'PC{i+1}' for i in range(optimal_k)])
    customer_data_pca.index = customer_data_scaled.index
    
    return customer_data_pca

customer_data_pca = scale_and_apply_pca(customer_data_cleaned)
print(f'\nPCA data shape: {customer_data_pca.shape}')
print('✓ Scaling & PCA Completed')

## 6. K-Means Clustering

In [ ]:
# Global variable to store kmeans for pkl saving later
kmeans = None

def apply_kmeans(pca_data, customer_data):
    global kmeans
    
    kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, max_iter=100, random_state=0)
    kmeans.fit(pca_data)
    
    clusters = kmeans.labels_
    customer_data_copy = customer_data.copy()
    customer_data_copy['cluster'] = clusters
    
    pca_data_copy = pca_data.copy()
    pca_data_copy['cluster'] = clusters
    
    return customer_data_copy, pca_data_copy

customer_data_clustered, customer_data_pca_clustered = apply_kmeans(customer_data_pca, customer_data_cleaned)

print('Cluster Distribution:')
print(customer_data_clustered['cluster'].value_counts().sort_index())
print('✓ Clustering Completed')

## 7. Evaluation Metrics

In [ ]:
def evaluate_clusters(pca_data):
    num_observations = len(pca_data)
    X = pca_data.drop('cluster', axis=1)
    clusters = pca_data['cluster']
    
    sil_score = silhouette_score(X, clusters)
    calinski_score = calinski_harabasz_score(X, clusters)
    davies_score = davies_bouldin_score(X, clusters)
    
    metrics = {
        'num_observations': num_observations,
        'silhouette_score': sil_score,
        'calinski_score': calinski_score,
        'davies_score': davies_score
    }
    return metrics

metrics = evaluate_clusters(customer_data_pca_clustered)

print('Cluster Evaluation Metrics:')
print(f'Number of Observations: {metrics["num_observations"]}')
print(f'Silhouette Score: {metrics["silhouette_score"]:.4f} (0-1, higher is better)')
print(f'Calinski Harabasz Score: {metrics["calinski_score"]:.4f} (higher is better)')
print(f'Davies Bouldin Score: {metrics["davies_score"]:.4f} (lower is better)')
print('✓ Evaluation Completed')

## 8. Generate Recommendations

In [ ]:
def generate_recommendations(raw_data, customer_data, outliers_data):
    outlier_customer_ids = outliers_data['CustomerID'].astype('float').unique()
    df_filtered = raw_data[~raw_data['CustomerID'].isin(outlier_customer_ids)]
    customer_data_copy = customer_data.copy()
    customer_data_copy['CustomerID'] = customer_data_copy['CustomerID'].astype('float')
    
    merged_data = df_filtered.merge(
        customer_data_copy[['CustomerID', 'cluster', 'Is_UK']],
        on='CustomerID',
        how='inner'
    )
    
    best_selling_products = merged_data.groupby(['cluster', 'StockCode', 'Description'])['Quantity'].sum().reset_index()
    best_selling_products = best_selling_products.sort_values(by=['cluster', 'Quantity'], ascending=[True, False])
    top_products_per_cluster = best_selling_products.groupby('cluster').head(10)
    
    best_products_by_country = merged_data.groupby(['Country', 'StockCode', 'Description'])['Quantity'].sum().reset_index()
    best_products_by_country = best_products_by_country.sort_values(by=['Country', 'Quantity'], ascending=[True, False])
    top_products_per_country = best_products_by_country.groupby('Country').head(10)
    
    customer_purchases = merged_data.groupby(['CustomerID', 'cluster', 'StockCode'])['Quantity'].sum().reset_index()
    customer_country = df_filtered.groupby('CustomerID')['Country'].first().reset_index()
    customer_country['CustomerID'] = customer_country['CustomerID'].astype('float')
    
    recommendations = []
    
    for cluster in top_products_per_cluster['cluster'].unique():
        top_products = top_products_per_cluster[top_products_per_cluster['cluster'] == cluster]
        customers_in_cluster = customer_data_copy[customer_data_copy['cluster'] == cluster][['CustomerID', 'Is_UK']]
        
        for _, customer_row in customers_in_cluster.iterrows():
            customer = customer_row['CustomerID']
            purchased_products = customer_purchases[(customer_purchases['CustomerID'] == customer) & (customer_purchases['cluster'] == cluster)]['StockCode'].tolist()
            
            not_purchased = top_products[~top_products['StockCode'].isin(purchased_products)]
            
            if len(not_purchased) < 3:
                country_row = customer_country[customer_country['CustomerID'] == customer]
                if not country_row.empty:
                    country = country_row['Country'].values[0]
                    country_products = top_products_per_country[top_products_per_country['Country'] == country]
                    country_not_purchased = country_products[~country_products['StockCode'].isin(purchased_products)]
                    not_purchased = pd.concat([not_purchased, country_not_purchased], ignore_index=True)
                    not_purchased = not_purchased.drop_duplicates(subset=['StockCode'])
            
            top_3 = not_purchased.head(3)
            
            if len(top_3) > 0:
                rec_values = top_3[['StockCode', 'Description']].values.flatten().tolist()
                while len(rec_values) < 6:
                    rec_values.extend(['', ''])
                recommendations.append([customer, cluster] + rec_values[:6])
            else:
                recommendations.append([customer, cluster, '', '', '', '', '', ''])
    
    recommendations_df = pd.DataFrame(
        recommendations,
        columns=['CustomerID', 'cluster', 'Rec1_StockCode', 'Rec1_Description', 'Rec2_StockCode', 'Rec2_Description', 'Rec3_StockCode', 'Rec3_Description']
    )
    
    final_df = customer_data_copy.merge(recommendations_df, on=['CustomerID', 'cluster'], how='right')
    return final_df

recommendations = generate_recommendations(df_clean, customer_data_clustered, outliers_data)
print(f'Recommendations generated: {recommendations.shape[0]} customers')
print('✓ Recommendation Generation Completed')
recommendations.head()

## 9. Visualizations

In [ ]:
# Cluster Distribution
plt.figure(figsize=(10, 5))
cluster_counts = customer_data_clustered['cluster'].value_counts().sort_index()
plt.bar(cluster_counts.index, cluster_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Customer Distribution Across Clusters')
for i, v in enumerate(cluster_counts.values):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Metrics Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(['Silhouette Score'], [metrics['silhouette_score']], color='#FF6B6B')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score')
axes[0].set_title(f'Silhouette Score\n({metrics["silhouette_score"]:.4f})')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(['Calinski Score'], [metrics['calinski_score']], color='#4ECDC4')
axes[1].set_ylabel('Score')
axes[1].set_title(f'Calinski Harabasz Score\n({metrics["calinski_score"]:.2f})')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(['Davies Bouldin Score'], [metrics['davies_score']], color='#45B7D1')
axes[2].set_ylabel('Score')
axes[2].set_title(f'Davies Bouldin Score\n({metrics["davies_score"]:.4f})')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 2D PCA Visualization of Clusters
pca_2d = PCA(n_components=2)
pca_2d_transformed = pca_2d.fit_transform(customer_data_pca)

plt.figure(figsize=(12, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for cluster in range(3):
    mask = customer_data_pca_clustered['cluster'] == cluster
    plt.scatter(pca_2d_transformed[mask, 0], pca_2d_transformed[mask, 1], 
               c=colors[cluster], label=f'Cluster {cluster}', alpha=0.6, s=50)

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Customer Clusters - PCA Visualization')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Summary & Key Insights

In [ ]:
print('='*60)
print('CUSTOMER SEGMENTATION PIPELINE - SUMMARY')
print('='*60)
print(f'\nTotal Customers Processed: {customer_data_clustered.shape[0]:,}')
print(f'Number of Clusters: 3')
print(f'\nCluster Sizes:')
for cluster in range(3):
    count = (customer_data_clustered['cluster'] == cluster).sum()
    pct = (count / len(customer_data_clustered)) * 100
    print(f'  Cluster {cluster}: {count:,} customers ({pct:.1f}%)')

print(f'\nClustering Quality Metrics:')
print(f'  Silhouette Score: {metrics["silhouette_score"]:.4f} (Fair separation)')
print(f'  Calinski Score: {metrics["calinski_score"]:.2f} (Well-separated groups)')
print(f'  Davies Bouldin Score: {metrics["davies_score"]:.4f} (Tight clusters)')

print(f'\nKey Features Engineered: 15')
print(f'  - Recency, Frequency, Monetary (RFM) features')
print(f'  - Temporal patterns (day of week, hour, purchase frequency)')
print(f'  - Geographic indicators (UK vs other countries)')
print(f'  - Transaction reliability (cancellation rate)')
print(f'  - Seasonal and trend analysis')

print(f'\nRecommendations Generated: {recommendations.shape[0]:,}')
print(f'  - Per customer: 3 top products')
print(f'  - Cluster-based: Yes')
print(f'  - Country-based fallback: Yes (for new users)')

print(f'\n✓ Pipeline Completed Successfully!')
print('='*60)

## 11. Save Models & Artifacts

In [ ]:
# Create models directory
os.makedirs('models', exist_ok=True)

# 1. Save Scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✓ scaler.pkl saved')

# 2. Save PCA
with open('models/pca.pkl', 'wb') as f:
    pickle.dump(pca, f)
print('✓ pca.pkl saved')

# 3. Save KMeans
with open('models/kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
print('✓ kmeans.pkl saved')

# 4. Save Recommendations dict {CustomerID: [{StockCode, Description} x3]}
recommendations_dict = {}
for _, row in recommendations.iterrows():
    customer_id = int(row['CustomerID'])
    recommendations_dict[customer_id] = [
        {'StockCode': row['Rec1_StockCode'], 'Description': row['Rec1_Description']},
        {'StockCode': row['Rec2_StockCode'], 'Description': row['Rec2_Description']},
        {'StockCode': row['Rec3_StockCode'], 'Description': row['Rec3_Description']},
    ]

with open('models/recommendations.pkl', 'wb') as f:
    pickle.dump(recommendations_dict, f)
print('✓ recommendations.pkl saved')

# 5. Save Cluster Top Products (fallback for new users)
cluster_top_products = {}
for cluster in range(3):
    cluster_recs = recommendations[recommendations['cluster'] == cluster]
    top = cluster_recs[['Rec1_StockCode', 'Rec1_Description']].rename(
        columns={'Rec1_StockCode': 'StockCode', 'Rec1_Description': 'Description'}
    ).drop_duplicates().head(10)
    cluster_top_products[cluster] = top.to_dict('records')

with open('models/cluster_top_products.pkl', 'wb') as f:
    pickle.dump(cluster_top_products, f)
print('✓ cluster_top_products.pkl saved')

# 6. Save Customer-Cluster mapping as CSV (for DB import)
customer_clusters = customer_data_clustered[['CustomerID', 'cluster']].copy()
customer_clusters.to_csv('models/customer_segments.csv', index=False)
print('✓ customer_segments.csv saved')

print('\n===== All Artifacts Saved =====')
print('Files in models/:')
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}') / 1024
    print(f'  {f} ({size:.1f} KB)')